# 34 孟加拉国 DIA 冻结候选统一 SIRIUS 补充注释

本 Notebook 对 Notebook 32 身份盲态冻结的全部 400 个跨极性簇执行统一 SIRIUS 注释，不按 Notebook 33 的实测谱库命中状态选择性处理。特征数、单电荷输入数和多电荷排除数均由本轮 manifest 动态确定。

流程分为三个阶段：

1. 在 Jupyter 中导出 Pos/Neg `.ms` 输入；
2. 在 SIRIUS 6 中运行 Molecular Formula Identification 和 CSI:FingerID；
3. 保持 SIRIUS 打开，回到本 Notebook 自动导入并冻结结果。

> 第一次不要直接 Run All。先运行到“导出输入”单元格，然后暂停并完成 SIRIUS 计算。内核使用 `env_ms`。

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'src' / 'pbt_msms').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'pbt_msms').exists(), '找不到项目根目录'
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
assert Path(sys.prefix).name.lower() == 'env_ms', f'请切换到 env_ms 内核；当前为 {sys.executable}'
print('ROOT =', ROOT)
print('Python =', sys.executable)


In [ ]:
from pbt_msms.bangladesh_dia_sirius import paths, sha256_file
p = paths(ROOT)
run33 = json.loads(p['run_manifest'].read_text(encoding='utf-8'))
assert run33['status'] == 'PUBLIC_LIBRARY_MATCHING_COMPLETE'
assert run33['counts']['frozen_clusters'] == 400
assert 0 <= run33['counts']['clusters_without_nonambiguous_library_match'] <= run33['counts']['frozen_clusters']
print('Notebook 33 frozen gate: PASS')


In [ ]:
import importlib
import pbt_msms.bangladesh_dia_sirius as bd_sirius
bd_sirius = importlib.reload(bd_sirius)  # 防止内核缓存旧版导出逻辑
export_manifest = bd_sirius.export_inputs(ROOT)
print(json.dumps(export_manifest, ensure_ascii=False, indent=2))
assert export_manifest['status'] == 'READY_FOR_SIRIUS_GUI_COMPUTATION'
assert export_manifest['n_clusters'] == 400
assert export_manifest['n_frozen_features'] == run33['counts']['case_spectra']
assert export_manifest['n_sirius_features'] + export_manifest['n_unsupported_multiply_charged_features'] == export_manifest['n_frozen_features']
print('导出完成。现在暂停 Notebook，转到下面的 SIRIUS 手工步骤。')


In [ ]:
# 本单元从刚生成的清单动态显示本轮 SIRIUS 数量和固定参数，避免硬编码漂移。
instructions_path = p['instructions']
assert instructions_path.is_file(), '请先运行上一个导出单元格'
print(instructions_path.read_text(encoding='utf-8'))
print('本轮导入数量：', export_manifest['mode_counts'])
print('多电荷保留审计：', export_manifest['n_unsupported_multiply_charged_features'])
print('完成上述 SIRIUS 步骤后，保持 SIRIUS 和两个项目打开，再继续运行下面的单元格。')

In [ ]:
# 完成上面的 SIRIUS 操作后，从这里继续。
port_file = Path.home() / '.sirius' / 'sirius-6.port'
assert port_file.is_file(), 'SIRIUS 6 没有运行。请启动 SIRIUS 并保持正负项目可用。'
print('SIRIUS REST port =', port_file.read_text(encoding='utf-8').strip())
print('SIRIUS running gate: PASS')



In [ ]:
import importlib
import pbt_msms.bangladesh_dia_sirius as bd_sirius
bd_sirius = importlib.reload(bd_sirius)
manifest = bd_sirius.import_results(ROOT, top_structure_n=10)
print(json.dumps(manifest, ensure_ascii=False, indent=2))
assert manifest['status'] == 'SIRIUS_ANNOTATIONS_IMPORTED_AND_FROZEN'
assert manifest['n_features'] == export_manifest['n_frozen_features']
assert manifest['n_clusters'] == export_manifest['n_clusters']


In [ ]:
import pandas as pd
OUT = p['out']
features = pd.read_csv(OUT / '34_feature_annotation_summary.csv.gz')
clusters = pd.read_csv(OUT / '34_cluster_annotation_summary.csv')
audit = pd.read_csv(OUT / '34_cohort_annotation_audit.csv')
assert len(features) == manifest['n_features'] and features['spectrum_id'].is_unique
assert len(clusters) == manifest['n_clusters'] and clusters['cross_polarity_cluster'].is_unique
assert audit['n_clusters'].sum() == manifest['n_clusters']
for filename, details in manifest['artifacts'].items():
    assert sha256_file(OUT / filename) == details['sha256'], f'哈希不一致：{filename}'
display(audit)
display(clusters[['cohort', 'cluster_rank', 'cross_polarity_cluster', 'annotation_level',
                  'molecular_formula', 'structure_name', 'inchikey_2d', 'csi_score',
                  'cross_member_annotation_conflict', 'cluster_annotation_status']]
        .sort_values(['cohort', 'cluster_rank']).groupby('cohort').head(10))
print(f"{manifest['n_clusters']} 个簇完整、Level 3/4 边界明确、输出哈希：全部 PASS")


## 完成判据

最后一个单元格必须显示：`400 个簇完整、Level 3/4 边界明确、输出哈希：全部 PASS`。

输出目录：`external_data/bangladesh_surface_water/06_revealed_evaluation/34_sirius_annotation_v1`。

Notebook 35 将比较四个冻结队列的注释覆盖率、结构类别和潜在高风险候选，但不会把 SIRIUS 候选当作标准品确证，也不会把它们当成 PBT 数值真值。